# Week 8 (Tue) - Class Activities in CS 315 (DuckDB Version)

Solving tasks from `netflix_data.db` using DuckDB SQL queries.

**Table of Contents**

0. [Q0: Connect to Database & List Tables](#q0)
1. [Q1: Inspect Table Data](#q1)
2. [Q2: Show Number of Rows](#q2)
3. [Q3: Show Column Names](#q3)
4. [Q4: Show Number of Total Countries](#q4)
5. [Q5: Show Number of TV Series](#q5)
6. [Q6: Number of TV Series in South Korea](#q6)
7. [Q7: Total Weeks of Squid Game in Italy](#q7)
8. [Q8: Time Frame for a Given Country](#q8)
9. [Q9: Is Number of Weeks in Ranking Equal?](#q9)
10. [Q10: Country with Fewest Weeks](#q10)

<a id="q0"></a>
### Q0: Connect to Database & List Tables

Connect to `netflix_data.db` and inspect the available tables in the database.

In [16]:
import duckdb

con = duckdb.connect('netflix_data.db')
con

When we open a connection it will remain open until we explicetely close it.

We can execute simple SQL commands as shown below.

In [17]:
con.sql("SHOW TABLES")

┌──────────┐
│   name   │
│ varchar  │
├──────────┤
│ films    │
│ tvseries │
└──────────┘

<a id="q1"></a>
### Q1: Load & Preview Data

Preview the top rows `tvseries`. This is similar to the `.head` method in pandas.

In [18]:
con.sql("SELECT * FROM tvseries LIMIT 5")

┌──────────────┬──────────────┬────────────┬─────────────┬──────────────────────┬───────────────────────────────────┬────────────────────────────┐
│ country_name │ country_iso2 │    week    │ weekly_rank │      show_title      │           season_title            │ cumulative_weeks_in_top_10 │
│   varchar    │   varchar    │  varchar   │    int64    │       varchar        │              varchar              │           int64            │
├──────────────┼──────────────┼────────────┼─────────────┼──────────────────────┼───────────────────────────────────┼────────────────────────────┤
│ Argentina    │ AR           │ 2026-10-04 │           1 │ Not a Stranger       │ Not a Stranger: Season 1          │                          3 │
│ Argentina    │ AR           │ 2026-10-04 │           2 │ My Sad Dead          │ My Sad Dead: Limited Series       │                          2 │
│ Argentina    │ AR           │ 2026-10-04 │           3 │ The Final Problem    │ The Final Problem: Limited Series │ 

<a id="q2"></a>
### Q2: Show number of rows

In [19]:
con.execute("SELECT COUNT(*) FROM tvseries")

Because the result is a table, if we want just the number, we will use the method `fetchone` to get the first row:

In [20]:
con.execute("SELECT COUNT(*) FROM tvseries").fetchone()

(256100,)

Since the result is a tuple, then we can index it like this:

In [21]:
con.sql("SELECT COUNT(*) FROM tvseries").fetchone()[0]

256100

<a id="q3"></a>
### Q3: Show column names

In [22]:
con.sql("DESCRIBE tvseries")

┌────────────────────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│        column_name         │ column_type │  null   │   key   │ default │  extra  │
│          varchar           │   varchar   │ varchar │ varchar │ varchar │ varchar │
├────────────────────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ country_name               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ country_iso2               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ week                       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ weekly_rank                │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ show_title                 │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ season_title               │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ cumulative_weeks_in_top_10 │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
└────────────────────────────┴─────────────┴─────────┴─────────┴─

<a id="q4"></a>
### Q4: Show number of total countries

In [23]:
con.sql("SELECT COUNT(DISTINCT country_name) FROM tvseries").fetchone()[0]

94

<a id="q5"></a>
### Q5: Show number of TV series

In [24]:
con.sql("SELECT COUNT(DISTINCT show_title) FROM tvseries").fetchone()[0]

3574

<a id="q6"></a>
### Q6: Number of TV series in South Korea

In [25]:
con.sql("""
    SELECT COUNT(DISTINCT show_title) 
    FROM tvseries 
    WHERE country_name = 'South Korea'
""").fetchone()[0]

481

<a id="q7"></a>
### Q7: Total weeks of Squid Game in Italy

In [26]:
con.sql("""
    SELECT COUNT(DISTINCT week) 
    FROM tvseries 
    WHERE country_name = 'Italy' AND show_title = 'Squid Game'
""").fetchone()[0]

29

<a id="q8"></a>
### Q8: Time frame for a given country

Write a function that given a country name prints out the start and end date of their rankings.

Once you have created the function, call it for a few countries to test it.

In [27]:
def show_timeframe(con, country):
    """A function to print out the time frame a country was in the Netflix rankings using DuckDB."""
    res = con.sql("""
        SELECT MIN(week), MAX(week)
        FROM (
            SELECT country_name, week FROM tvseries
        )
        WHERE country_name = $1
    """, params=[country]).fetchone()
    
    start, end = res[0], res[1]
    print(f"Netflix rankings for {country} exist between {start} and {end}.")

show_timeframe(con, "Chile")
show_timeframe(con, "Russia")

Netflix rankings for Chile exist between 2021-07-04 and 2026-10-04.
Netflix rankings for Russia exist between 2021-07-04 and 2022-02-27.


<a id="q9"></a>
### Q9: Is number of weeks equal for each country?

Check if total rows match `20 * unique_weeks * unique_countries`.

In [28]:
is_equal = con.sql("""
    SELECT COUNT(*) = 10 * COUNT(DISTINCT week) * COUNT(DISTINCT country_name) 
    FROM tvseries
""").fetchone()[0]

is_equal

False

We should have expected this to be `False` since we saw in Q8 that Russia stopped being in the rankings after it attacked Ukraine in February 2022.

<a id="q10"></a>
### Q10: Country with fewest weeks

Find the country with the fewest weeks in the Netflix rankings.

In [29]:
con.sql("""
SELECT country_name, COUNT(DISTINCT week) AS weekCnt
FROM tvseries
GROUP BY country_name
ORDER BY weekCnt ASC
LIMIT 5;""")

┌──────────────┬─────────┐
│ country_name │ weekCnt │
│   varchar    │  int64  │
├──────────────┼─────────┤
│ Russia       │      35 │
│ New Zealand  │     275 │
│ Norway       │     275 │
│ Nicaragua    │     275 │
│ Oman         │     275 │
└──────────────┴─────────┘

In [ ]:
conn = duckdb.connect()
conn.sql("INSTALL sqlite; LOAD sqlite;")
conn.sql("ATTACH 'sql vs pandas/school_database.db' AS school (TYPE sqlite);")
conn.sql("USE school;")

conn.sql("""
SELECT student_name, gpa
FROM students
WHERE grade_level = 10 AND gpa > 3
ORDER BY gpa DESC;
""")


┌──────────────┬────────┐
│ student_name │  gpa   │
│   varchar    │ double │
├──────────────┼────────┤
│ Liam Green   │    4.0 │
│ Abby Johnson │    3.1 │
└──────────────┴────────┘

In [32]:
conn.sql("""
SELECT
    s.student_name,
    COUNT(sg.class_id) AS num_classes
FROM
    students AS s
LEFT JOIN
    student_grades AS sg
ON
    s.id = sg.student_id
GROUP BY
    s.id, s.student_name;
""")

┌─────────────────┬─────────────┐
│  student_name   │ num_classes │
│     varchar     │    int64    │
├─────────────────┼─────────────┤
│ Daniel Brown    │           4 │
│ Eva Martinez    │           4 │
│ Henry Taylor    │           4 │
│ Olivia Adams    │           4 │
│ Isabella Moore  │           4 │
│ Bob Smith       │           4 │
│ Frank Wilson    │           4 │
│ Grace Lee       │           4 │
│ Peter Park      │           4 │
│ Catherine Davis │           4 │
│ Abby Johnson    │           4 │
│ Mia Harris      │           4 │
│ Jack Thompson   │           4 │
│ Karen White     │           4 │
│ Liam Green      │           4 │
│ Noah Scott      │           0 │
└─────────────────┴─────────────┘
  16 rows             2 columns

In [37]:
connn = duckdb.connect()
connn.execute("ATTACH 'simple_netflix.db' AS nf (TYPE sqlite)")
connn.sql("SHOW TABLES") 
connn.sql("USE nf;")
connn.sql("SELECT * FROM netflix")


┌───────┬──────────────┬────────────────────────────────┬──────────┬────────────┬─────────────┐
│  id   │ country_name │           show_title           │ category │    week    │ weekly_rank │
│ int64 │   varchar    │            varchar             │ varchar  │  varchar   │    int64    │
├───────┼──────────────┼────────────────────────────────┼──────────┼────────────┼─────────────┤
│     1 │ Hong Kong    │ When Life Gives You Tangerines │ TV       │ 2025-04-13 │           3 │
│     2 │ Hong Kong    │ Hospital Playlist              │ TV       │ 2021-10-10 │           9 │
│     3 │ Hong Kong    │ Anna                           │ TV       │ 2021-10-03 │           2 │
└───────┴──────────────┴────────────────────────────────┴──────────┴────────────┴─────────────┘